# Figure Skating Judging Bias: Exploratory Analysis

This notebook provides a reproducible starting point for examining individual judge marks. It does not establish that bias exists. Differences and outliers are descriptive signals that require validation and context.

## Research questions

- Do GOE and PCS marks differ by entry nationality after accounting for competition and scoring context?
- Do competitors with stronger prior results receive more generous judge marks, especially on attempts with technical-call flags?
- Do judges' marks differ when the judge and entry represent the same country?
- Which judge marks depart from the rest of their panel, and are those differences consistent?

The four event categories are men's singles, women's singles, pair skating, and ice dance. Pairs and ice dance are team events, not separate men's and women's competitions; use one stable entry/team ID and the country represented by that team. Analyze categories separately because their marks, elements, and components are not directly interchangeable.

Keep statistical anomalies, systematic scoring differences, potential conflicts of interest, and proven intentional manipulation separate. This notebook can only assess patterns in the data supplied; it cannot establish intent.

## Get or refresh the data

From a terminal at the project root, activate `.venv` and run:

```bash
python -m figure_skating_bias.collectors.isu_results
```

This collects official per-skater protocols from European and Four Continents Championships (2022–2026), World Championships (2022 and 2024–2026), and the 2022 and 2026 Winter Olympics. It writes the normalized data to `data/processed/judge_marks.csv`. Use `--refresh` to redownload sources; otherwise cached originals are reused. The 2023 World Championships is excluded because its older combined-PDF format is not yet supported. The collector preserves judge nationality exactly as the official panel reports it. If that field says `ISU`, the judge's home-country nationality is not provided by that source and cannot be tested until separately verified.

## Input data contract

Expected file: `data/processed/judge_marks.csv`, with **one row per individual judge mark** and these columns:

| Column | Meaning |
|---|---|
| `competition_id`, `competition_name`, `season`, `scoring_rule_version` | Competition identity and applicable scoring rules |
| `event_category` | `men_singles`, `women_singles`, `pair_skating`, or `ice_dance` |
| `segment` | Short program or free skating |
| `entry_id`, `entry_name`, `entry_nationality` | Stable skater or team identity and represented country |
| `judge_id`, `judge_nationality` | Judge identity and the official panel's nationality label; `ISU` is not a home country |
| `judge_federation` | Listed member federation from the seasonal officials list; this is not necessarily citizenship and may be unknown |
| `judge_federation_match`, `judge_federation_source_url` | Match status and source provenance; only `exact` matches are used for same-nationality analysis |
| `mark_type` | `GOE` or `PCS` |
| `element_type` | Element category for GOE marks; nullable for PCS |
| `component` | PCS component for PCS marks; nullable for GOE |
| `score` | This judge's mark |
| `panel_mean`, `panel_median` | Source panel summaries; the deviation analysis below recomputes leave-one-judge-out references from individual marks |
| `source_url`, `source_file` | Provenance for the source protocol |

Country codes should be normalized consistently before comparing them. For pairs and ice dance, `entry_nationality` is the country represented in the event; keep member-level identities in a separate roster table if athlete-level questions are needed. PCS component sets and rules can change between seasons, so retain the applicable scoring-rule version and do not assume component scores are comparable across rule changes. This notebook assumes the input has already been parsed and validated against the applicable official rules. Confirm that protocols, calls, panel membership, high/low treatment, deductions, and reconstructed official totals/ranks are correct before interpreting any model or counterfactual. Keep source documents immutable and retain their provenance.

In [1]:
import importlib.util
from pathlib import Path

import polars as pl

PROJECT_ROOT = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "pyproject.toml").is_file()
        and (folder / "src/figure_skating_bias").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from within the figure_skating_bias project.")
DATA_PATH = PROJECT_ROOT / "data/processed/judge_marks.csv"
REQUIRED_COLUMNS = {
    "competition_id", "competition_name", "season", "scoring_rule_version",
    "event_category", "segment",
    "entry_id", "entry_name", "entry_nationality",
    "judge_id", "judge_nationality", "judge_federation", "judge_federation_match",
    "mark_type", "element_type",
    "component", "score", "panel_mean", "panel_median",
    "source_url", "source_file",
}
NON_NULL_COLUMNS = {
    "competition_id", "season", "scoring_rule_version", "event_category",
    "segment", "entry_id",
    "entry_nationality", "judge_id", "judge_nationality",
    "mark_type", "score", "panel_mean", "panel_median",
    "source_url", "source_file",
}

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"No judge-mark dataset found at {DATA_PATH}. Collect and validate official "
        "protocols, then write the normalized CSV using the data contract above."
    )

marks = pl.read_csv(DATA_PATH, try_parse_dates=True)
missing_columns = REQUIRED_COLUMNS - set(marks.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

null_counts = marks.select(
    pl.col(column).null_count().alias(column) for column in sorted(NON_NULL_COLUMNS)
).row(0, named=True)
null_columns = {column: count for column, count in null_counts.items() if count}
if null_columns:
    raise ValueError(f"Required fields contain null values: {null_columns}")

if marks.is_empty():
    raise ValueError(f"Dataset at {DATA_PATH} contains no judge marks.")

invalid_mark_types = (
    marks.select(pl.col("mark_type").unique())
    .to_series()
    .drop_nulls()
    .to_list()
)
unexpected_mark_types = set(invalid_mark_types) - {"GOE", "PCS"}
if unexpected_mark_types:
    raise ValueError(f"Unexpected mark_type values: {sorted(unexpected_mark_types)}")

expected_event_categories = {"men_singles", "women_singles", "pair_skating", "ice_dance"}
actual_event_categories = set(marks.get_column("event_category").unique().to_list())
unexpected_event_categories = actual_event_categories - expected_event_categories
if unexpected_event_categories:
    raise ValueError(
        f"Unexpected event_category values: {sorted(unexpected_event_categories)}. "
        f"Expected values from {sorted(expected_event_categories)}."
    )

invalid_detail_rows = marks.filter(
    ((pl.col("mark_type") == "GOE") & pl.col("element_type").is_null())
    | ((pl.col("mark_type") == "PCS") & pl.col("component").is_null())
)
if invalid_detail_rows.height:
    raise ValueError(
        f"{invalid_detail_rows.height} rows are missing element_type for GOE "
        "or component for PCS."
    )

print(f"Loaded {marks.height:,} judge marks from {DATA_PATH}")


Loaded 854,048 judge marks from /Users/nadin/Documents/youtube_stuff/figure_skating_unfair/data/processed/judge_marks.csv


## Data coverage and integrity checks

Review coverage before analysis. The tables below do not replace protocol-level validation. Confirm that GOE rows represent individual element marks, PCS rows represent a single component mark, and the panel summaries are calculated over the correct judges and applicable scoring rules.

In [2]:
coverage = marks.group_by("competition_id", "event_category").agg(
    pl.col("competition_name").first(),
    pl.col("season").first(),
    pl.col("entry_id").n_unique().alias("entries_or_teams"),
    pl.col("judge_id").n_unique().alias("judges"),
    pl.len().alias("judge_marks"),
).sort("season", "competition_name", "event_category")
display(coverage)

missing_categories = expected_event_categories - actual_event_categories
if missing_categories:
    print(f"No data yet for event categories: {sorted(missing_categories)}")

display(
    marks.group_by("event_category", "mark_type", "segment")
    .agg(pl.len().alias("rows"), pl.col("entry_id").n_unique().alias("entries_or_teams"))
    .sort("event_category", "mark_type", "segment")
)

display(
    marks.group_by("event_category", "mark_type")
    .agg(
        pl.col("score").min().alias("minimum"),
        pl.col("score").median().alias("median"),
        pl.col("score").mean().alias("mean"),
        pl.col("score").max().alias("maximum"),
    )
)


competition_id,event_category,competition_name,season,entries_or_teams,judges,judge_marks
str,str,str,str,u32,u32,u32
"""ec2022""","""ice_dance""","""European Figure Skating Champi…","""season2122""",26,11,4518
"""ec2022""","""men_singles""","""European Figure Skating Champi…","""season2122""",33,10,6687
"""ec2022""","""pair_skating""","""European Figure Skating Champi…","""season2122""",21,12,4203
"""ec2022""","""women_singles""","""European Figure Skating Champi…","""season2122""",35,12,6570
"""fc2022""","""ice_dance""","""Four Continents Figure Skating…","""season2122""",10,7,1568
…,…,…,…,…,…,…
"""owg2026""","""women_singles""","""Olympic Winter Games 2026""","""season2526""",29,13,5814
"""wc2026""","""ice_dance""","""World Figure Skating Champions…","""season2526""",31,13,4311
"""wc2026""","""men_singles""","""World Figure Skating Champions…","""season2526""",36,13,6444


event_category,mark_type,segment,rows,entries_or_teams
str,str,str,u32,u32
"""ice_dance""","""GOE""","""free_dance""",67794,190
"""ice_dance""","""GOE""","""rhythm_dance""",42235,206
"""ice_dance""","""PCS""","""free_dance""",23459,190
"""ice_dance""","""PCS""","""rhythm_dance""",26873,206
"""men_singles""","""GOE""","""free_skating""",121067,302
…,…,…,…,…
"""pair_skating""","""PCS""","""short_program""",18959,141
"""women_singles""","""GOE""","""free_skating""",130389,436
"""women_singles""","""GOE""","""short_program""",83999,456


event_category,mark_type,minimum,median,mean,maximum
str,str,f64,f64,f64,f64
"""ice_dance""","""GOE""",-5.0,2.0,1.951513,5.0
"""ice_dance""","""PCS""",0.25,7.75,7.591373,10.0
"""men_singles""","""PCS""",0.25,7.25,7.184645,10.0
"""men_singles""","""GOE""",-5.0,1.0,0.552054,5.0
"""women_singles""","""PCS""",0.5,7.0,6.760518,10.0
"""women_singles""","""GOE""",-5.0,1.0,0.479192,5.0
"""pair_skating""","""PCS""",0.5,7.0,7.077467,10.0
"""pair_skating""","""GOE""",-5.0,1.0,0.635447,5.0


## Judge deviations from panel summaries

The notebook recomputes leave-one-judge-out panel means and medians from the individual scores, so a judge's mark is not part of its own reference value. Rows with only one available judge mark have no leave-one-out reference and are excluded from these summaries. A positive deviation means the judge's mark is above the other judges' reference; a negative deviation means it is below. Interpret this descriptively, and compare like with like (event category, competition, segment, mark type, element/component) rather than treating raw GOE and PCS marks or marks across event categories as interchangeable.

In [3]:
import numpy as np
import pandas as pd

comparison_keys = [
    "competition_id", "event_category", "segment", "mark_type", "entry_id",
    "element_type", "component",
]
score_data = pd.DataFrame(
    marks.select(*comparison_keys, "score").to_dict(as_series=False)
)
score_groups = score_data.groupby(comparison_keys, sort=False, dropna=False)["score"]
panel_counts = score_groups.transform("size")
score_data["panel_mean_loo"] = (
    score_groups.transform("sum") - score_data["score"]
) / (panel_counts - 1)

def leave_one_out_medians(scores: pd.Series) -> pd.Series:
    values = scores.to_numpy()
    return pd.Series(
        [np.median(np.delete(values, i)) for i in range(len(values))]
        if len(values) > 1 else [np.nan],
        index=scores.index,
    )

score_data["panel_median_loo"] = score_groups.transform(leave_one_out_medians)
marks = marks.with_columns(
    pl.Series("panel_mean_loo", score_data["panel_mean_loo"].to_list()),
    pl.Series("panel_median_loo", score_data["panel_median_loo"].to_list()),
)
deviations = marks.with_columns(
    (pl.col("score") - pl.col("panel_mean_loo")).alias("deviation_from_panel_mean"),
    (pl.col("score") - pl.col("panel_median_loo")).alias("deviation_from_panel_median"),
).filter(pl.col("deviation_from_panel_mean").is_finite())

judge_summary = deviations.group_by(
    "judge_id", "judge_nationality", "competition_id", "event_category", "mark_type"
).agg(
    pl.len().alias("marks"),
    pl.col("deviation_from_panel_mean").mean().alias("mean_deviation"),
    pl.col("deviation_from_panel_mean").median().alias("median_deviation"),
    pl.col("deviation_from_panel_median").median().alias("median_vs_panel_median"),
    pl.col("deviation_from_panel_mean").std().alias("deviation_sd"),
).sort("competition_id", "mark_type", "mean_deviation")
display(judge_summary)

print(
    "Judge-level deviations are descriptive only; the source label ISU does not "
    "identify a judge's home-country nationality."
)


judge_id,judge_nationality,competition_id,event_category,mark_type,marks,mean_deviation,median_deviation,median_vs_panel_median,deviation_sd
str,str,str,str,str,u32,f64,f64,f64,f64
"""ETTRUP_KIM_LOUISE""","""DEN""","""csbud2022""","""women_singles""","""GOE""",360,-0.193464,-0.125,0.0,0.593954
"""CURRIE_DEBORAH""","""USA""","""csbud2022""","""men_singles""","""GOE""",339,-0.161136,-0.125,0.0,0.638887
"""MILUSE_SAFRANKOVA""","""CZE""","""csbud2022""","""women_singles""","""GOE""",589,-0.139521,0.0,0.0,0.514998
"""CURRIE_DEBORAH""","""USA""","""csbud2022""","""women_singles""","""GOE""",360,-0.13703,-0.125,0.0,0.553002
"""GRAMPELHUBER_STEAFAN""","""AUT""","""csbud2022""","""ice_dance""","""GOE""",267,-0.127341,0.0,0.0,0.888296
…,…,…,…,…,…,…,…,…,…
"""BESCHEA_MARINA""","""ISU""","""wc2026""","""women_singles""","""PCS""",171,0.227339,0.21875,0.25,0.373404
"""AHISKAL_SENEM""","""ISU""","""wc2026""","""men_singles""","""PCS""",72,0.230469,0.234375,0.25,0.365017
"""HAN_KYUNG_YOUNG""","""ISU""","""wc2026""","""men_singles""","""PCS""",108,0.237847,0.25,0.25,0.41902


Judge-level deviations are descriptive only; the source label ISU does not identify a judge's home-country nationality.


In [4]:
if importlib.util.find_spec("matplotlib") is None:
    print("Optional plot skipped. Install it with: python -m pip install -e '.[visualization]'")
else:
    import matplotlib.pyplot as plt

    plot_data = judge_summary.filter(pl.col("marks") > 0).sort("mean_deviation")
    labels = [f"{event} / {judge} ({country}) / {competition} / {kind}"
              for event, judge, country, competition, kind in plot_data.select(
                  "event_category", "judge_id", "judge_nationality", "competition_id", "mark_type"
              ).iter_rows()]
    values = plot_data.get_column("mean_deviation").to_list()
    fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(labels))))
    ax.barh(labels, values)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Mean mark minus panel mean")
    ax.set_title("Descriptive judge deviations by competition and mark type")
    fig.tight_layout()
    plt.show()


Optional plot skipped. Install it with: python -m pip install -e '.[visualization]'


## Exploratory adjusted comparisons

The models below fit each event category and mark type separately so incomparable scores are not pooled. They use OLS with combined competition-by-segment-by-element/component scoring-context fixed effects. Judge fixed effects are omitted because judges are nested within event panels; judge-level deviations are summarized separately above. These are exploratory associations, not causal estimates. They do not control for all performance quality, technical calls, order effects, reputation, or rule changes; marks may be dependent within entry/team, competition, and panel. Clustered standard errors by entry/team address only one source of dependence. Treat uncertainty cautiously, especially with few entries or competitions; p-values are unadjusted across the many nationality comparisons, and no coefficient establishes favoritism.

The models are fit separately for GOE and PCS because the two mark types have different scales and meanings. GOE models compare like element categories; PCS models compare like components. To keep this notebook responsive, scoring-context fixed effects are absorbed by within-context demeaning rather than expanded into thousands of dummy columns; this is algebraically equivalent to including those fixed effects in OLS. PCS definitions/scales and scoring rules can change across seasons, so include rule-version checks and avoid interpreting cross-season component scores as directly equivalent.

In [5]:
import pandas as pd
import patsy
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.linalg import qr
from scipy.stats import norm
from statsmodels.stats.sandwich_covariance import cov_cluster

model_columns = [
    "score", "entry_id", "entry_nationality", "event_category",
    "competition_id", "segment", "mark_type", "element_type", "component",
]
model_data = pd.DataFrame(marks.select(model_columns).to_dict(as_series=False))
model_data["entry_nationality"] = model_data["entry_nationality"].str.strip().str.upper()
model_data["mark_category"] = (
    model_data["mark_type"] + ":" + model_data["element_type"].fillna(model_data["component"])
)
model_data["scoring_context"] = (
    model_data["competition_id"] + "|" + model_data["segment"] + "|" +
    model_data["mark_category"]
)

class ClusteredOLSResult:
    def __init__(self, params: pd.Series, covariance: pd.DataFrame):
        self.params = params
        self._covariance = covariance

    def cov_params(self) -> pd.DataFrame:
        return self._covariance

def fit_clustered_ols(formula: str, data: pd.DataFrame):
    """Fit clustered OLS after absorbing scoring-context fixed effects."""
    cluster_count = data["entry_id"].nunique()
    if cluster_count < 2:
        raise ValueError(
            "Need at least 2 distinct entries/teams for clustered inference; "
            f"found {cluster_count}."
        )
    if cluster_count < 30:
        print(
            f"Caution: only {cluster_count} entry/team clusters; cluster-robust "
            "standard errors may be unreliable."
        )
    outcome, rhs = formula.split("~", maxsplit=1)
    terms = [term.strip() for term in rhs.split("+")]
    fixed_effect_terms = [term for term in terms if term == "C(scoring_context)"]
    if len(fixed_effect_terms) != 1:
        raise ValueError("Expected exactly one C(scoring_context) fixed effect.")
    predictor_terms = [term for term in terms if term != "C(scoring_context)"]
    design = patsy.dmatrix(" + ".join(predictor_terms), data, return_type="dataframe")
    if "Intercept" in design:
        design = design.drop(columns="Intercept")

    fixed_effect = data["scoring_context"]
    y = data[outcome.strip()].astype(float)
    y_within = y - y.groupby(fixed_effect, sort=False).transform("mean")
    x_within = design - design.groupby(fixed_effect, sort=False).transform("mean")
    matrix = x_within.to_numpy(dtype=float)
    if matrix.shape[1] == 0:
        raise ValueError("No predictor variation remains after absorbing fixed effects.")

    r_factor, pivots = qr(matrix, mode="r", pivoting=True)
    diagonal = abs(r_factor.diagonal())
    tolerance = max(matrix.shape) * 2.220446049250313e-16 * diagonal.max()
    rank = int((diagonal > tolerance).sum())
    if rank == 0:
        raise ValueError("Predictors are not estimable after absorbing fixed effects.")
    if rank < matrix.shape[1]:
        retained = sorted(pivots[:rank])
        x_within = x_within.iloc[:, retained]
        matrix = x_within.to_numpy(dtype=float)

    model = sm.OLS(y_within.to_numpy(dtype=float), matrix).fit(method="qr")
    cluster_codes = pd.factorize(data["entry_id"], sort=False)[0]
    covariance = cov_cluster(model, cluster_codes, use_correction=False)
    context_count = fixed_effect.nunique()
    full_parameter_count = int(model.df_model) + context_count - 1
    observation_count = len(data)
    if observation_count <= full_parameter_count:
        raise ValueError("Not enough observations for the fixed-effects model.")
    correction = (
        cluster_count / (cluster_count - 1)
        * (observation_count - 1) / (observation_count - full_parameter_count)
    )
    covariance *= correction
    parameters = pd.Series(model.params, index=x_within.columns)
    covariance_frame = pd.DataFrame(
        covariance, index=x_within.columns, columns=x_within.columns
    )
    return ClusteredOLSResult(parameters, covariance_frame)

base_formula = "score ~ C(entry_nationality) + C(scoring_context)"

def coefficient_summary(model, terms):
    """Summarize selected terms without taking square roots of unrelated covariances."""
    estimates = model.params.loc[terms]
    covariance = model.cov_params().loc[terms, terms]
    standard_errors = np.sqrt(np.diag(covariance))
    if not np.isfinite(standard_errors).all() or (standard_errors <= 0).any():
        raise ValueError("Selected effects have non-estimable clustered standard errors.")
    z_scores = estimates.to_numpy() / standard_errors
    critical_value = norm.ppf(0.975)
    return pd.DataFrame(
        {
            "estimate": estimates,
            "std_error": standard_errors,
            "p_value_normal": 2 * norm.sf(np.abs(z_scores)),
            "ci_lower_95": estimates.to_numpy() - critical_value * standard_errors,
            "ci_upper_95": estimates.to_numpy() + critical_value * standard_errors,
        },
        index=terms,
    ).round(3)

def nationality_effects(model):
    """Show entry-nationality contrasts against the model's reference country."""
    terms = model.params.index[
        model.params.index.str.startswith("C(entry_nationality)")
    ]
    return coefficient_summary(model, terms)
base_model_data = model_data.dropna(subset=[
    "score", "entry_id", "entry_nationality", "event_category",
    "scoring_context",
])
base_models = {}
for (event_category, mark_type), category_data in base_model_data.groupby(
    ["event_category", "mark_type"], sort=True
):
    model_key = (event_category, mark_type)
    base_models[model_key] = fit_clustered_ols(base_formula, category_data)
    print(f"\n=== {event_category} / {mark_type} ===")
    display(nationality_effects(base_models[model_key]))



=== ice_dance / GOE ===


,estimate,std_error,p_value_normal,ci_lower_95,ci_upper_95
C(entry_nationality)[T.AUS],0.064,0.283,0.821,-0.490,0.618
C(entry_nationality)[T.AUT],-0.172,0.251,0.494,-0.664,0.321
C(entry_nationality)[T.AZE],-0.922,0.251,0.000,-1.415,-0.429
C(entry_nationality)[T.BEL],-0.862,0.247,0.000,-1.346,-0.378
C(entry_nationality)[T.BIH],-1.019,0.254,0.000,-1.517,-0.520
C(entry_nationality)[T.BLR],-0.673,0.295,0.023,-1.252,-0.094
C(entry_nationality)[T.BRA],-0.695,0.236,0.003,-1.158,-0.232
C(entry_nationality)[T.CAN],1.064,0.277,0.000,0.521,1.606
C(entry_nationality)[T.CHN],-0.169,0.257,0.512,-0.672,0.335
C(entry_nationality)[T.CYP],-0.172,0.245,0.482,-0.651,0.307



=== ice_dance / PCS ===


,estimate,std_error,p_value_normal,ci_lower_95,ci_upper_95
C(entry_nationality)[T.AUS],0.132,0.500,0.791,-0.848,1.112
C(entry_nationality)[T.AUT],-0.295,0.397,0.458,-1.073,0.484
C(entry_nationality)[T.AZE],-1.177,0.398,0.003,-1.957,-0.397
C(entry_nationality)[T.BEL],-1.042,0.390,0.008,-1.808,-0.277
C(entry_nationality)[T.BIH],-1.378,0.402,0.001,-2.165,-0.591
C(entry_nationality)[T.BLR],-0.961,0.382,0.012,-1.709,-0.212
C(entry_nationality)[T.BRA],-0.916,0.384,0.017,-1.668,-0.163
C(entry_nationality)[T.CAN],1.325,0.428,0.002,0.486,2.163
C(entry_nationality)[T.CHN],-0.114,0.420,0.787,-0.938,0.710
C(entry_nationality)[T.CYP],-0.345,0.383,0.368,-1.095,0.406



=== men_singles / GOE ===


,estimate,std_error,p_value_normal,ci_lower_95,ci_upper_95
C(entry_nationality)[T.ARM],-0.903,0.132,0.000,-1.161,-0.644
C(entry_nationality)[T.AUS],-0.535,0.243,0.027,-1.010,-0.059
C(entry_nationality)[T.AUT],-0.607,0.139,0.000,-0.880,-0.334
C(entry_nationality)[T.AZE],0.199,0.103,0.053,-0.003,0.400
C(entry_nationality)[T.BEL],0.125,0.261,0.631,-0.386,0.637
C(entry_nationality)[T.BLR],-0.689,0.163,0.000,-1.009,-0.369
C(entry_nationality)[T.BUL],-0.698,0.206,0.001,-1.102,-0.294
C(entry_nationality)[T.CAN],0.425,0.177,0.017,0.077,0.772
C(entry_nationality)[T.CHN],0.060,0.130,0.646,-0.195,0.314
C(entry_nationality)[T.CRO],0.222,0.117,0.057,-0.007,0.451


... (6 outputs left out)


=== women_singles / GOE ===


,estimate,std_error,p_value_normal,ci_lower_95,ci_upper_95
C(entry_nationality)[T.ARG],-1.259,0.405,0.002,-2.054,-0.465
C(entry_nationality)[T.AUS],-1.007,0.368,0.006,-1.728,-0.286
C(entry_nationality)[T.AUT],-0.340,0.392,0.386,-1.108,0.428
C(entry_nationality)[T.AZE],-0.547,0.355,0.123,-1.243,0.148
C(entry_nationality)[T.BEL],0.509,0.386,0.186,-0.246,1.265
...,...,...,...,...,...
C(entry_nationality)[T.TPE],-0.714,0.383,0.062,-1.464,0.036
C(entry_nationality)[T.TUR],-1.575,0.337,0.000,-2.237,-0.914
C(entry_nationality)[T.UKR],-1.093,0.332,0.001,-1.744,-0.441
C(entry_nationality)[T.USA],0.379,0.337,0.260,-0.281,1.038



=== women_singles / PCS ===


,estimate,std_error,p_value_normal,ci_lower_95,ci_upper_95
C(entry_nationality)[T.ARG],-1.569,0.579,0.007,-2.703,-0.434
C(entry_nationality)[T.AUS],-1.060,0.617,0.086,-2.269,0.149
C(entry_nationality)[T.AUT],-0.257,0.576,0.655,-1.387,0.872
C(entry_nationality)[T.AZE],-0.309,0.626,0.622,-1.536,0.918
C(entry_nationality)[T.BEL],0.845,0.614,0.169,-0.359,2.049
...,...,...,...,...,...
C(entry_nationality)[T.TPE],-1.018,0.643,0.113,-2.278,0.241
C(entry_nationality)[T.TUR],-1.725,0.538,0.001,-2.779,-0.671
C(entry_nationality)[T.UKR],-1.097,0.544,0.044,-2.163,-0.030
C(entry_nationality)[T.USA],0.836,0.539,0.121,-0.220,1.892


## Same-nationality association

This model adds an indicator for a judge and entry/team sharing a normalized country code, fitted separately for each event category and mark type. A combined competition-by-segment-by-element/component fixed effect controls for scoring context. Judge fixed effects are omitted because they are nested in competition panels. The same-country coefficient is an adjusted association among observed marks, not proof of favoritism. It uses only exact judge-federation matches from seasonal officials lists; missing or ambiguous matches are excluded, and a listed federation is not necessarily citizenship. A category without verified judge-federation data and both match types cannot estimate the association.

In [6]:
same_country_data = pd.DataFrame(marks.select(
    "score", "entry_id", "entry_nationality", "judge_federation",
    "judge_federation_match", "judge_id",
    "competition_id", "event_category", "segment", "mark_type", "element_type", "component",
).to_dict(as_series=False))
same_country_data["entry_nationality"] = (
    same_country_data["entry_nationality"].str.strip().str.upper()
)
same_country_data["judge_federation"] = (
    same_country_data["judge_federation"].str.strip().str.upper()
)
same_country_data["mark_category"] = (
    same_country_data["mark_type"] + ":" +
    same_country_data["element_type"].fillna(same_country_data["component"])
)
same_country_data["scoring_context"] = (
    same_country_data["competition_id"] + "|" + same_country_data["segment"] + "|" +
    same_country_data["mark_category"]
)
display(
    same_country_data.groupby(["judge_federation_match", "judge_federation"], dropna=False)
    .size().rename("marks_by_judge_federation_match_status")
)
known_country_data = same_country_data[
    same_country_data["judge_federation_match"].eq("exact")
    & same_country_data["judge_federation"].notna()
    & ~same_country_data["judge_federation"].isin(["", "UNKNOWN"])
].copy()
if known_country_data.empty:
    print(
        "Same-nationality analysis not estimated: no exact judge-federation "
        "matches are available in the selected data."
    )
known_country_data["same_nationality"] = (
    known_country_data["entry_nationality"] == known_country_data["judge_federation"]
).astype(int)

if not known_country_data.empty:
    display(
        known_country_data.groupby("same_nationality", observed=True)
        .agg(marks=("score", "size"), entries_or_teams=("entry_id", "nunique"))
    )

same_country_formula = "score ~ same_nationality + C(scoring_context)"
same_country_model_data = known_country_data.dropna(subset=[
    "score", "entry_id", "entry_nationality", "judge_federation",
    "event_category", "scoring_context",
])
same_country_models = {}
for (event_category, mark_type), category_data in same_country_model_data.groupby(
    ["event_category", "mark_type"], sort=True
):
    if category_data["same_nationality"].nunique() < 2:
        print(
            f"{event_category} / {mark_type}: not estimable; only one "
            "same-nationality value is present."
        )
        continue
    model_key = (event_category, mark_type)
    same_country_models[model_key] = fit_clustered_ols(same_country_formula, category_data)
    print(f"\n=== {event_category} / {mark_type} ===")
    display(coefficient_summary(same_country_models[model_key], ["same_nationality"]))


judge_federation_match  judge_federation
exact                   AUS                   9346
                        AUT                   6082
                        BEL                   4092
                        BUL                   5058
                        CAN                  17443
                        CHN                  11177
                        CRO                   2044
                        CYP                   9386
                        DEN                    710
                        ESP                   1052
                        EST                  11337
                        FIN                  14636
                        FRA                  22551
                        GBR                   7300
                        GEO                   9788
                        GER                  13499
                        HKG                    613
                        HUN                   9765
                        ISL              

,marks,entries_or_teams
same_nationality,,
0,315118,757
1,27201,470



=== ice_dance / GOE ===


,estimate,std_error,p_value_normal,ci_lower_95,ci_upper_95
same_nationality,0.324,0.04,0.0,0.246,0.402



=== ice_dance / PCS ===


,estimate,std_error,p_value_normal,ci_lower_95,ci_upper_95
same_nationality,0.321,0.066,0.0,0.192,0.45


... (8 outputs left out)


=== women_singles / GOE ===


,estimate,std_error,p_value_normal,ci_lower_95,ci_upper_95
same_nationality,0.18,0.023,0.0,0.136,0.224



=== women_singles / PCS ===


,estimate,std_error,p_value_normal,ci_lower_95,ci_upper_95
same_nationality,0.132,0.038,0.001,0.058,0.207


## Prior success and PCS: exploratory reputation proxy

The dataset has no independent measure of reputation. This section uses a competitor's best segment rank in the previous calendar year as a limited proxy: top six versus outside the top six. For GOE, it compares panel-mean marks on elements with at least one notation marker, controlling for event/segment, base value, and the marker combination. For PCS, it compares current mean marks by component, controlling for event/segment/component context, the entry's mean GOE, and the share of marked elements. Uncertainty is clustered by competition and entry, and Holm adjustment covers all eight category-by-mark tests.

These are associations, **not evidence that a name causes higher scores**. Previous success also reflects ability, artistry, coaching, and unmeasured performance; GOE and protocol calls do not fully measure quality. The top-six proxy only uses prior competitions present in this dataset, so first-year observations have no history and are omitted. `q`, `<`, `!`, `e`, and `F` are parsed as protocol notation markers, not as a complete or independently verified error record. Ice dance has very few marked-element attempts, so its GOE estimate is especially uncertain.

In [7]:
import re

from scipy.stats import t as student_t
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.sandwich_covariance import cov_cluster_2groups

reputation_columns = [
    "competition_id", "season", "event_category", "segment", "entry_id", "rank",
    "mark_type", "component", "element_number", "element_type", "base_value", "score",
]
reputation_data = pd.DataFrame(
    marks.select(*reputation_columns).to_dict(as_series=False)
)
reputation_data["event_year"] = (
    reputation_data["season"].str[-2:].astype(int) + 2000
)

entry_keys = ["competition_id", "event_category", "segment", "entry_id"]
entries = reputation_data[
    entry_keys + ["season", "rank", "event_year"]
].drop_duplicates()
prior_results = (
    entries.groupby(
        ["event_category", "segment", "entry_id", "event_year"],
        as_index=False,
    )["rank"]
    .min()
    .rename(columns={"event_year": "year", "rank": "prior_best_rank"})
)
prior_results["year"] += 1
entries = entries.merge(
    prior_results,
    left_on=["event_category", "segment", "entry_id", "event_year"],
    right_on=["event_category", "segment", "entry_id", "year"],
    how="left",
).drop(columns="year")
entries["prior_top6"] = entries["prior_best_rank"].le(6).astype(int)

def protocol_call_flags(value):
    text = value or ""
    flags = (
        ("q", bool(re.search(r"(?<![A-Z])q(?=$|[ +,])", text))),
        ("underrotation", "<" in text),
        ("edge", "!" in text),
        ("edge_call", bool(re.search(r"(?<=[A-Za-z0-9])e(?=$|[ +,])", text))),
        ("fall", "F" in text.split()),
    )
    return tuple(name for name, present in flags if present)

goe_marks = reputation_data[reputation_data["mark_type"].eq("GOE")].copy()
goe_marks["call_pattern"] = goe_marks["element_type"].map(protocol_call_flags)
goe_marks["has_call"] = goe_marks["call_pattern"].map(bool)
goe_attempts = (
    goe_marks.groupby(entry_keys + ["element_number"], as_index=False)
    .agg(
        element_type=("element_type", "first"),
        base_value=("base_value", "first"),
        panel_goe=("score", "mean"),
        call_pattern=("call_pattern", "first"),
        has_call=("has_call", "first"),
    )
)
technical_context = goe_attempts.groupby(entry_keys, as_index=False).agg(
    mean_goe=("panel_goe", "mean"),
    call_rate=("has_call", "mean"),
)
pcs_by_component = (
    reputation_data[reputation_data["mark_type"].eq("PCS")]
    .groupby(entry_keys + ["component"], as_index=False)["score"]
    .mean()
    .rename(columns={"score": "pcs_score"})
)
pcs_analysis = (
    pcs_by_component.merge(technical_context, on=entry_keys)
    .merge(
        entries[entry_keys + ["prior_best_rank", "prior_top6"]],
        on=entry_keys,
    )
    .dropna(subset=["prior_best_rank"])
)
pcs_analysis["context"] = (
    pcs_analysis["competition_id"] + "|" + pcs_analysis["segment"] + "|"
    + pcs_analysis["component"]
)

goe_analysis = (
    goe_attempts[goe_attempts["has_call"]]
    .merge(
        entries[entry_keys + ["prior_best_rank", "prior_top6"]],
        on=entry_keys,
    )
    .dropna(subset=["prior_best_rank"])
)
goe_analysis["context"] = (
    goe_analysis["competition_id"] + "|" + goe_analysis["segment"]
)
goe_analysis["call_pattern"] = goe_analysis["call_pattern"].map(
    lambda flags: "+".join(flags)
)

reputation_results = []
models_to_fit = [
    (
        "GOE",
        goe_analysis,
        "panel_goe ~ prior_top6 + base_value + C(context) + C(call_pattern)",
    ),
    (
        "PCS",
        pcs_analysis,
        "pcs_score ~ prior_top6 + mean_goe + call_rate + C(context)",
    ),
]
for mark_type, analysis_data, formula in models_to_fit:
    for category in sorted(analysis_data["event_category"].unique()):
        category_data = analysis_data[analysis_data["event_category"].eq(category)]
        model = smf.ols(formula, data=category_data).fit()
        competition_cluster = np.asarray(
            pd.factorize(category_data["competition_id"])[0], dtype=np.int64
        )
        entry_cluster = np.asarray(
            pd.factorize(category_data["entry_id"])[0], dtype=np.int64
        )
        covariance, _, _ = cov_cluster_2groups(
            model, competition_cluster, entry_cluster
        )
        coefficient_index = model.model.exog_names.index("prior_top6")
        variance = float(covariance[coefficient_index, coefficient_index])
        if not np.isfinite(variance) or variance <= 0:
            raise ValueError(
                f"Invalid two-way clustered variance for {mark_type}/{category}: {variance}"
            )
        standard_error = np.sqrt(variance)
        degrees_of_freedom = category_data["competition_id"].nunique() - 1
        critical_value = student_t.ppf(0.975, degrees_of_freedom)
        estimate = float(model.params["prior_top6"])
        test_statistic = estimate / standard_error
        p_value = 2 * student_t.sf(abs(test_statistic), degrees_of_freedom)
        reputation_results.append(
            {
                "mark_type": mark_type,
                "event_category": category,
                "observations": len(category_data),
                "entries": category_data["entry_id"].nunique(),
                "competitions": category_data["competition_id"].nunique(),
                "prior_top6_estimate": estimate,
                "ci_lower_95": estimate - critical_value * standard_error,
                "ci_upper_95": estimate + critical_value * standard_error,
                "p_value": p_value,
            }
        )

reputation_summary = pd.DataFrame(reputation_results)
reputation_summary["p_value_holm"] = multipletests(
    reputation_summary["p_value"], method="holm"
)[1]
display(
    reputation_summary[
        [
            "mark_type", "event_category", "observations", "entries", "competitions",
            "prior_top6_estimate", "ci_lower_95", "ci_upper_95", "p_value_holm",
        ]
    ].round(3)
)

/var/folders/_8/fkwn5mq515qf1fjs_dw872l40000gn/T/ipykernel_42940/638400583.py:117: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = smf.ols(formula, data=category_data).fit()


,mark_type,event_category,observations,entries,competitions,prior_top6_estimate,ci_lower_95,ci_upper_95,p_value_holm
0,GOE,ice_dance,87,48,36,0.653,0.071,1.236,0.058
1,GOE,men_singles,1902,133,57,0.151,0.044,0.258,0.019
2,GOE,pair_skating,526,59,54,0.140,-0.064,0.345,0.175
3,GOE,women_singles,2535,164,57,0.341,0.217,0.465,0.000
4,PCS,ice_dance,3510,92,55,0.182,0.107,0.257,0.000
5,PCS,men_singles,4806,142,57,0.453,0.341,0.566,0.000
6,PCS,pair_skating,2355,62,55,0.428,0.263,0.594,0.000
7,PCS,women_singles,4440,168,57,0.414,0.322,0.506,0.000


## Counterfactual rankings: not implemented yet

Do not adjust official rankings by simply subtracting a regression coefficient, removing an arbitrary judge, or replacing marks with medians. A defensible counterfactual must first reproduce the applicable ISU rules for each season and event, including element-level aggregation, dropped high/low marks where applicable, segment totals, deductions, tie-breaking, and ranking. Implement and independently test that reconstruction against official protocols/results before simulating any changes. Preserve actual results separately and label every simulated result as counterfactual.

## Before reporting findings

- [ ] Verify every source URL and retain a source file/checksum manifest.
- [ ] Validate judge identities/nationalities and entry/team identities/represented countries for the competition date.
- [ ] Manually verify normalized entry IDs and aliases, especially team changes and athletes who change federation.
- [ ] Confirm pair and ice-dance team members and represented country in a separate roster source where relevant.
- [ ] Reconstruct official scores and rankings under the correct season's rules.
- [ ] Check duplicated, missing, miscoded, and extreme values against the original protocol.
- [ ] Analyze GOE and each PCS component separately as well as in justified combined models.
- [ ] Account for technical difficulty/calls, performance quality, deductions, order, event, and rule changes where data permits.
- [ ] Test robustness, report effect sizes and confidence intervals, and address multiple comparisons.
- [ ] Distinguish statistical significance from practical relevance; do not claim intent.

The current collection covers 16 official championship/Olympic events from 2022–2026. The 2023 World Championships remains excluded because its legacy combined-PDF format is not supported; officials-list affiliations are unavailable for 2021/22 through 2023/24. To distinguish reputation from ability/artistry, the next data improvement is independently validated performance-quality annotations or a credible pre-event reputation measure, not just more scores.